In [134]:
import numpy as np
import pandas as pd

import pandas as pd
import numpy as np
import re #आपल्याला special characters आणि numbers काढायचे आहेत.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [135]:
resumes=pd.read_csv('Resume.csv')

In [136]:
jobs=pd.read_csv('job_sample.csv')

In [12]:
resumes.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 52.3 MB


In [137]:
jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   country          22000 non-null  str  
 1   country_code     22000 non-null  str  
 2   date_added       122 non-null    str  
 3   has_expired      22000 non-null  str  
 4   job_board        22000 non-null  str  
 5   job_description  22000 non-null  str  
 6   job_title        22000 non-null  str  
 7   job_type         20372 non-null  str  
 8   location         22000 non-null  str  
 9   organization     15133 non-null  str  
 10  page_url         22000 non-null  str  
 11  salary           3446 non-null   str  
 12  sector           16806 non-null  str  
 13  uniq_id          22000 non-null  str  
dtypes: str(14)
memory usage: 67.1 MB


In [138]:
print("Job Shape :", jobs.shape)
print("Resume Shape :", resumes.shape)

Job Shape : (22000, 14)
Resume Shape : (2484, 4)


In [139]:
print(jobs.columns)

print(resumes.columns)

Index(['country', 'country_code', 'date_added', 'has_expired', 'job_board',
       'job_description', 'job_title', 'job_type', 'location', 'organization',
       'page_url', 'salary', 'sector', 'uniq_id'],
      dtype='str')
Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')


In [140]:
#Keep Only Required Columns
# We only need the job_title and job_description columns 
# because the recommendation system compares the resume text with the job description. 
# Other columns like salary, country, and page URL are not required for similarity matching.

jobs = jobs[['job_title','job_description']]

In [141]:
resumes = resumes[['Resume_str','Category']]

In [142]:
jobs.head()

,job_title,job_description
0,IT Support Technician Job in Madison,TeamSoft is seeing an IT Support Specialist to...
1,Business Reporter/Editor Job in Madison,The Wisconsin State Journal is seeking a flexi...
2,Johnson & Johnson Family of Companies Job Appl...,Report this job About the Job DePuy Synthes Co...
3,Engineer - Quality Job in Dixon,Why Join Altec? If you’re considering a career...
4,Shift Supervisor - Part-Time Job in Camphill,Position ID# 76162 # Positions 1 State CT C...


In [143]:
resumes.head()

,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [144]:
jobs.isnull().sum()

job_title          0
job_description    0
dtype: int64

In [145]:
resumes.isnull().sum()

Resume_str    0
Category      0
dtype: int64

In [146]:
print(jobs.shape)

print(resumes.shape)

(22000, 2)
(2484, 2)


In [147]:
#Text Cleaning


def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\\S+',' ',text)

    text = re.sub(r'[^a-zA-Z ]',' ',text)

    text = re.sub(r'\\s+',' ',text)

    return text

In [148]:
#Clean Job Description

jobs['job_description'] = jobs['job_description'].apply(clean_text)
jobs['job_title'] = jobs['job_title'].apply(clean_text)

In [149]:
resumes['Resume_str'] = resumes['Resume_str'].apply(clean_text)
resumes['Category'] = resumes['Category'].apply(clean_text)

In [150]:
jobs.head()

,job_title,job_description
0,it support technician job in madison,teamsoft is seeing an it support specialist to...
1,business reporter editor job in madison,the wisconsin state journal is seeking a flexi...
2,johnson johnson family of companies job appl...,report this job about the job depuy synthes co...
3,engineer quality job in dixon,why join altec if you re considering a career...
4,shift supervisor part time job in camphill,position id positions state ct c...


In [48]:
resumes.head()

,Resume_str,Category
0,hr administrator marketing associate ...,hr
1,hr specialist us hr operations ...,hr
2,hr director summary over ...,hr
3,hr specialist summary dedica...,hr
4,hr manager skill highlights ...,hr


In [151]:
#Combine Job Title + Job Description

jobs['job_text'] = jobs['job_title'] + " " + jobs['job_description']

In [152]:
jobs.head()

,job_title,job_description,job_text
0,it support technician job in madison,teamsoft is seeing an it support specialist to...,it support technician job in madison teamsoft ...
1,business reporter editor job in madison,the wisconsin state journal is seeking a flexi...,business reporter editor job in madison the wi...
2,johnson johnson family of companies job appl...,report this job about the job depuy synthes co...,johnson johnson family of companies job appl...
3,engineer quality job in dixon,why join altec if you re considering a career...,engineer quality job in dixon why join altec...
4,shift supervisor part time job in camphill,position id positions state ct c...,shift supervisor part time job in camphill p...


In [153]:
jobs['job_text'].str.len().describe()

count    22000.000000
mean      2620.423500
std       1722.318916
min         42.000000
25%       1434.000000
50%       2285.000000
75%       3394.000000
max      20249.000000
Name: job_text, dtype: float64

In [154]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [155]:
#Convert Job Text into Vectors

job_vectors = tfidf.fit_transform(job['job_text'])

In [156]:
job_vectors.shape

(22000, 90757)

In [157]:
#Resume ला TF-IDF मध्ये Convert करणे
resume_text = resumes['Resume_str'][0]


In [158]:
print(resume_text)

         hr administrator marketing associate  hr administrator       summary     dedicated customer service manager with     years of experience in hospitality and customer service management    respected builder and leader of customer focused teams  strives to instill a shared  enthusiastic commitment to customer service          highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multi tasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq      micros    opera pms     fidelio    opera    reservation system  ors      holidex    completed courses and seminars in customer service  sales strategies  inventory control  loss preve

In [159]:
#create resume vector
resume_vector = tfidf.transform([resume_text])

In [160]:
resume_vector.shape

(1, 90757)

In [161]:
#Cosine Similarity
#Resume ला सर्व 22000 jobs सोबत compare करेल.
similarity_scores = cosine_similarity(resume_vector, job_vectors)

In [162]:
similarity_scores.shape

(1, 22000)

In [163]:
#Top 5 Jobs Find
top_indices = similarity_scores[0].argsort()[-5:][::-1]


In [164]:
#Display Recommended Jobs
recommended_jobs= jobs.iloc[top_indices]

recommended_jobs[['job_title','job_description']]


,job_title,job_description
1461,events public relations assistant job in orl...,we have an immediate need for a public retail ...
16958,entry level assistant marketing advertising ...,about us the job window is seeking a entry lev...
8375,marketing manager entry level job in aurora,marketing manager entry levelour expanding co...
10891,public relations communications assistant ...,the job window has an immediate need for a pub...
7851,customer service client relations associate ...,do you have experience in the restaurant reta...


In [165]:
#Create Job Recommendation Function

def recommend_job(resume_text):

    # Step 1: Clean Resume Text
    resume_text = clean_text(resume_text)

    # Step 2: Convert Resume into TF-IDF vector
    resume_vector = tfidf.transform([resume_text])

    # Step 3: Calculate similarity with all jobs
    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    )

    # Step 4: Get top 5 matching jobs
    top_indices = similarity_scores[0].argsort()[-5:][::-1]

    # Step 5: Display jobs
    recommended_job = job.iloc[top_indices]

    return recommended_job[['job_title','job_description']]


In [166]:
import pdfplumber
def extract_resume_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + " "

    return text

In [167]:
import os

os.listdir("uploads")

['.ipynb_checkpoints', 'prachi resume.pdf (1).pdf']

In [168]:
pdf_path = "uploads/prachi resume.pdf (1).pdf"

In [169]:
resume_text = extract_resume_text(pdf_path)

print(resume_text)

Name: Prachi
CareerObjective:
Looking foropportunitiesin softwaredevelopmentanddata science.
Skills:
Python
SQL
MachineLearning
Natural LanguageProcessing
Pandas
NumPy
Data Analysis
HTML
CSS
Java
Projects:
Job Recommendation Systemusing NLPandMachineLearning.
Education:
Diploma ComputerEngineering
Experience:
Internship in Artificial IntelligenceandMachineLearning. 


In [171]:
recommended_jobs = recommend_jobs(resume_text)

recommended_jobs

,job_title,job_description
3660,java programmer with javascript and html job i...,experis is hiring a backend java programmer fo...
6815,monster,title java python developerlocation middle...
21492,marketing analyst job in cincinnati,email marketing analyst html css we are see...
13777,qa automation engineer selenium python java jo...,responsibilities kforce is working with a well...
18158,javascript html css ui developer job in irving,javascript html css ui developerdetailslocat...


In [172]:
recommend_jobs(resume_text)

,job_title,job_description
3660,java programmer with javascript and html job i...,experis is hiring a backend java programmer fo...
6815,monster,title java python developerlocation middle...
21492,marketing analyst job in cincinnati,email marketing analyst html css we are see...
13777,qa automation engineer selenium python java jo...,responsibilities kforce is working with a well...
18158,javascript html css ui developer job in irving,javascript html css ui developerdetailslocat...


In [173]:
#Improve Recommendation

skills = [
    'python',
    'sql',
    'machine learning',
    'nlp',
    'pandas',
    'numpy',
    'java',
    'html',
    'css'
]


In [174]:
def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in skills:
        if skill in text:
            found_skills.append(skill)

    return found_skills

In [175]:
resume_skills = extract_skills(resume_text)

resume_skills

['python', 'sql', 'nlp', 'pandas', 'numpy', 'java', 'html', 'css']

In [183]:
import pickle

with open("tfidf.pkl","wb") as file:
    pickle.dump(tfidf,file)

In [184]:
with open("job_vectors.pkl","wb") as file:
    pickle.dump(job_vectors,file)

In [185]:
jobs.to_pickle("jobs.pkl")